<a href="https://colab.research.google.com/github/LIBY70/Data-Analysis/blob/main/ida_week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab7: scipy.stats를 활용한 통계 기초

본 실습 자료는 『배워서 바로 써먹는 데이터 분석 with 파이썬』 (설진욱, 생능북스)의 내용을 참고하여 제작되었습니다.

## 0. 라이브러리 불러오기

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib as mpl
import matplotlib.pyplot as plt

### 한글 폰트 설정

In [ ]:
!apt-get install -y fonts-nanum

In [ ]:
import platform

if platform.system() == 'Darwin':       # macOS
    mpl.rc('font', family='AppleGothic')
elif platform.system() == 'Windows':    # Windows
    mpl.rc('font', family='Malgun Gothic')
else:                                   # Linux / Colab
    import matplotlib.font_manager as fm
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    mpl.rc('font', family='NanumGothic')

mpl.rc('axes', unicode_minus=False)     # 마이너스 기호 깨짐 방지
print('폰트 설정 완료:', mpl.rcParams['font.family'])

---
## PART 1. 기술통계 (Descriptive Statistics)

데이터의 특징을 수치로 요약하는 방법.  
중심 경향(평균, 중앙값, 최빈값), 산포도(분산, 표준편차, IQR), 분포 형태(왜도, 첨도)로 구성.

### 1-1. 데이터 준비

학생 30명의 시험 점수 데이터 생성.

In [ ]:
# 재현성을 위해 난수 시드 고정
np.random.seed(42)

# 학생 30명의 시험 점수
scores = np.array([58, 62, 65, 65, 67, 68, 70, 70, 70, 72,
                   73, 74, 74, 75, 75, 76, 77, 78, 78, 80,
                   81, 82, 83, 85, 85, 87, 88, 90, 92, 95])

scores

In [ ]:
len(scores)

### 1-2. 중심 경향 (Measures of Central Tendency)

데이터의 중심 위치를 나타내는 값.
- **평균 (Mean)**: 모든 값의 합 ÷ 개수
- **중앙값 (Median)**: 정렬했을 때 정가운데 값
- **최빈값 (Mode)**: 가장 자주 등장하는 값

In [ ]:
# 평균
mean_val = np.mean(scores)
mean_val

In [ ]:
# 중앙값
median_val = np.median(scores)
median_val

In [ ]:
# 최빈값 (scipy.stats.mode)
mode_result = stats.mode(scores, keepdims=True)
print(f"최빈값 (Mode):  {mode_result.mode[0]} (빈도: {mode_result.count[0]}회)")

### 1-3. 산포도 (Measures of Dispersion)

데이터가 얼마나 퍼져 있는지를 나타내는 값.
- **분산 (Variance)**: 편차 제곱의 평균 (`ddof=1`이면 **표본분산**)
- **표준편차 (Std Dev)**: 분산의 제곱근, 원래 단위와 동일
- **범위 (Range)**: 최댓값 - 최솟값
- **IQR**: 3사분위수(Q3) - 1사분위수(Q1), 이상치에 강건함

In [ ]:
# 분산 (ddof=1: 표본분산, ddof=0: 모분산)
np.var(scores, ddof=1)

In [ ]:
# 표준편차
np.std(scores, ddof=1)

In [ ]:
# 범위
scores.max() - scores.min()

In [ ]:
# IQR (Interquartile Range)
q1 = np.percentile(scores, 25)
q3 = np.percentile(scores, 75)
iqr_val = stats.iqr(scores)
print(f"Q1 (25%): {q1}")
print(f"Q3 (75%): {q3}")
print(f"IQR (Q3 - Q1): {iqr_val}")

### 1-4. 분포 형태 (Shape of Distribution)

- **왜도 (Skewness)**: 분포의 비대칭 정도
  - 양수 → 오른쪽 꼬리가 긴 분포 (right-skewed)
  - 음수 → 왼쪽 꼬리가 긴 분포 (left-skewed)
- **첨도 (Kurtosis)**: 분포의 뾰족함 정도 (기준: 0, 정규분포)
  - 양수 → 정규분포보다 뾰족 (leptokurtic)
  - 음수 → 정규분포보다 납작 (platykurtic)

In [ ]:
# 왜도 (Skewness)
skew_val = stats.skew(scores)
skew_val

In [ ]:
if skew_val > 0:
    print("오른쪽으로 치우친 분포 (right-skewed)")
elif skew_val < 0:
    print("왼쪽으로 치우친 분포 (left-skewed)")
else:
    print("대칭 분포")

In [ ]:
# 첨도 (Kurtosis)
kurt_val = stats.kurtosis(scores)
kurt_val

### 1-5. pandas describe()

`describe()`로 기술통계 수치를 한 번에 요약.

In [ ]:
df = pd.DataFrame({'시험점수': scores})
summary = df.describe()
summary

### 1-6. 시각화 — 히스토그램 & 박스플롯

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 히스토그램 — 분포 확인
axes[0].hist(scores, bins=8, color='steelblue', edgecolor='black')
axes[0].axvline(mean_val, color='red', linestyle='--', label=f'평균={mean_val:.1f}')
axes[0].axvline(median_val, color='orange', linestyle='--', label=f'중앙값={median_val:.1f}')
axes[0].set_title('시험 점수 분포 (히스토그램)')
axes[0].set_xlabel('점수')
axes[0].set_ylabel('빈도')
axes[0].legend()

# 박스플롯 — IQR 및 이상치 확인
axes[1].boxplot(scores, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('시험 점수 분포 (박스플롯)')
axes[1].set_ylabel('점수')
axes[1].set_xticks([1])
axes[1].set_xticklabels(['시험점수'])

plt.tight_layout()
plt.show()

---
## PART 2. 구간추정 (Confidence Interval)

모수(모평균 등)의 참값이 포함될 것으로 기대하는 구간 추정.  
- 모분산을 **아는** 경우 → **z 분포** 사용  
- 모분산을 **모르는** 경우 → **t 분포** 사용 (소표본일수록 중요)

### 2-1. 모분산을 아는 경우 — z 분포

신뢰구간 공식: $\bar{x} \pm z_{\alpha/2} \cdot \dfrac{\sigma}{\sqrt{n}}$

In [ ]:
# 시나리오: 모표준편차(sigma)가 10점으로 알려져 있다고 가정
x_bar = np.mean(scores)   # 표본평균
sigma = 10                # 모표준편차 (알고 있다고 가정)
n = len(scores)           # 표본 크기
alpha = 0.05              # 유의수준 (95% 신뢰구간)

# z 임계값: 양측이므로 alpha/2
z_critical = stats.norm.ppf(1 - alpha / 2)  # 1.96 (95% 기준)
print(f"z 임계값 (95%): {z_critical:.4f}")

# 신뢰구간 계산
margin = z_critical * (sigma / np.sqrt(n))
ci_lower = x_bar - margin
ci_upper = x_bar + margin

print(f"표본평균: {x_bar:.2f}")
print(f"95% 신뢰구간: ({ci_lower:.2f}, {ci_upper:.2f})")
# → 이 구간이 모평균을 포함할 확률이 95%

### 2-2. 모분산을 모르는 경우 — t 분포

소표본이거나 모분산을 모를 때 t 분포 사용.  
자유도(df) = n - 1

**예제**: 건전지 10개의 수명(시간)을 측정했다.

In [ ]:
# 건전지 수명 데이터 (단위: 시간)
battery = np.array([502, 498, 505, 511, 497, 508, 500, 496, 503, 510])
n_bat = len(battery)
x_bar_bat = np.mean(battery)
s_bat = np.std(battery, ddof=1)   # 표본표준편차
alpha = 0.05

print(f"표본 크기:    n = {n_bat}")
print(f"표본평균:     x̄ = {x_bar_bat:.2f} 시간")
print(f"표본표준편차: s = {s_bat:.2f} 시간")

# t 임계값 (95%, 자유도 = n-1 = 9)
t_critical = stats.t.ppf(1 - alpha / 2, df=n_bat - 1)
print(f"t 임계값 (df={n_bat-1}, 95%): {t_critical:.4f}")

# 방법 1: 직접 계산
margin_t = t_critical * (s_bat / np.sqrt(n_bat))
print(f"[방법 1: 직접 계산]")
print(f"95% 신뢰구간: ({x_bar_bat - margin_t:.2f}, {x_bar_bat + margin_t:.2f})")

# 방법 2: stats.t.interval() 한 줄 계산
ci = stats.t.interval(0.95, df=n_bat - 1, loc=x_bar_bat, scale=stats.sem(battery))
print(f"[방법 2: stats.t.interval()]")
print(f"95% 신뢰구간: ({ci[0]:.2f}, {ci[1]:.2f})")

### 2-3. 신뢰수준 변화에 따른 구간 비교

신뢰수준이 높아질수록 구간 폭 증가. (더 확실하게 포함하려면 범위를 넓혀야 함)

In [ ]:
# 건전지 데이터로 90%, 95%, 99% 신뢰구간 비교
confidence_levels = [0.90, 0.95, 0.99]

print(f"신뢰수준	하한	상한	구간 폭")
print("-" * 40)
for cl in confidence_levels:
    ci = stats.t.interval(cl, df=n_bat - 1, loc=x_bar_bat, scale=stats.sem(battery))
    width = ci[1] - ci[0]
    print(f"  {int(cl*100)}%	{ci[0]:.2f}	{ci[1]:.2f}	{width:.2f}")

---
## PART 3. 가설검정 기초

**가설검정 절차**
1. 귀무가설(H₀)과 대립가설(H₁) 설정
2. 유의수준($\alpha$) 설정 (보통 0.05)
3. 검정통계량 계산
4. $p$값 확인 → **$p \leq \alpha$** 이면 H₀ 기각

**p값 판단 기준**
- **$p \leq \alpha$** → H₀ 기각 (통계적으로 유의함)
- **$p > \alpha$** → H₀ 기각 실패 (유의한 차이 없음)

### 3-1. 단일 표본 t-검정 (One-sample t-test)

시나리오: **"우리 반 평균 점수가 전국 평균(70점)과 다른가?"**

- H₀: 우리 반 평균 = 70 (차이 없음)
- H₁: 우리 반 평균 ≠ 70 (차이 있음, 양측검정)

In [ ]:
pop_mean = 70  # 전국 평균 (귀무가설의 기준값)

t_stat, p_value = stats.ttest_1samp(scores, popmean=pop_mean)

print(f"단일 표본 t-검정 결과:")
print(f"  t-통계량: {t_stat:.4f}")
print(f"  p-값:     {p_value:.4f}")
print()

# 판단
alpha = 0.05
if p_value <= alpha:
    print(f"p={p_value:.4f} ≤ α={alpha}: H₀ 기각")
    print("우리 반 평균은 전국 평균(70점)과 통계적으로 유의한 차이가 있음")
else:
    print(f"p={p_value:.4f} > α={alpha}: H₀ 기각 실패")
    print("우리 반 평균이 전국 평균(70점)과 다르다고 할 수 없음")

### 3-2. 양측 vs 단측 검정

- `alternative='two-sided'` : H₁: μ ≠ 70 (양측)
- `alternative='greater'`   : H₁: μ > 70 (단측, 오른쪽)
- `alternative='less'`      : H₁: μ < 70 (단측, 왼쪽)

In [ ]:
for alt in ['two-sided', 'greater', 'less']:
    t, p = stats.ttest_1samp(scores, popmean=70, alternative=alt)
    print(f"alternative='{alt}'\t: t={t:6.3f}, p={p:.4f}", end="")
    print(f"  → {'기각' if p <= 0.05 else '기각 실패'}")

### 3-3. 독립 표본 t-검정 (Independent t-test)

시나리오: **"A반과 B반의 평균 점수에 차이가 있는가?"**

- H₀: A반 평균 = B반 평균
- H₁: A반 평균 ≠ B반 평균

In [ ]:
# A반, B반 점수 데이터
group_a = np.array([72, 68, 75, 80, 65, 78, 70, 74, 69, 77,
                    73, 71, 76, 66, 79])
group_b = np.array([65, 60, 68, 72, 58, 70, 63, 67, 62, 71,
                    64, 66, 69, 57, 74])

print(f"A반 평균: {np.mean(group_a):.2f}, 표준편차: {np.std(group_a, ddof=1):.2f}")
print(f"B반 평균: {np.mean(group_b):.2f}, 표준편차: {np.std(group_b, ddof=1):.2f}")
print()

# 독립 표본 t-검정 (등분산 가정)
t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=True)
print(f"[등분산 t-검정 (equal_var=True)]")
print(f"  t-통계량: {t_stat:.4f}, p-값: {p_value:.4f}")
print(f"  → {'H₀ 기각: 두 반의 평균에 유의한 차이가 있음' if p_value <= 0.05 else 'H₀ 기각 실패'}")
print()

# Welch t-검정 (등분산 가정 없음)
t_stat_w, p_value_w = stats.ttest_ind(group_a, group_b, equal_var=False)
print(f"[Welch t-검정 (equal_var=False)]")
print(f"  t-통계량: {t_stat_w:.4f}, p-값: {p_value_w:.4f}")
print(f"  → {'H₀ 기각' if p_value_w <= 0.05 else 'H₀ 기각 실패'}")

### 3-4. 대응 표본 t-검정 (Paired t-test)

시나리오: **"학원 수강 전후 점수에 차이가 있는가?"**  
같은 학생의 전/후 점수를 비교 → 대응 표본

- H₀: 수강 전후 평균 차이 = 0
- H₁: 수강 전후 평균 차이 ≠ 0

In [ ]:
# 같은 학생 10명의 학원 수강 전후 점수
before = np.array([60, 55, 70, 65, 58, 72, 63, 68, 50, 75])
after  = np.array([68, 62, 75, 71, 65, 78, 70, 72, 58, 80])

diff = after - before
print(f"점수 향상량 (after - before): {diff}")
print(f"평균 향상량: {np.mean(diff):.2f}점")
print()

# 대응 표본 t-검정
t_stat, p_value = stats.ttest_rel(before, after)

print(f"대응 표본 t-검정 결과:")
print(f"  t-통계량: {t_stat:.4f}")
print(f"  p-값:     {p_value:.4f}")
print()

if p_value <= 0.05:
    print(f"→ p={p_value:.4f} <= 0.05: H₀ 기각")
    print("  학원 수강 전후 점수 차이가 통계적으로 유의합니다.")
else:
    print(f"→ p={p_value:.4f} ≥ 0.05: H₀ 기각 실패")
    print("  수강 전후 차이가 통계적으로 유의하지 않습니다.")

---
## PART 4. 카이제곱 검정 (Chi-square Test)

범주형 데이터에 사용하는 검정.
- **적합도 검정**: 관측 빈도가 기대 빈도와 다른가?
- **독립성 검정**: 두 범주형 변수 사이에 연관이 있는가?

### 4-1. 적합도 검정 (Goodness of Fit)

시나리오: **"주사위가 공평한가? (각 면이 나올 확률이 1/6인가?)"**

- H₀: 각 면이 나올 확률은 동일 (공평한 주사위)
- H₁: 확률이 동일하지 않음 (불공평한 주사위)

In [ ]:
# 주사위를 60번 던진 결과 (관측 빈도)
observed = np.array([8, 12, 9, 11, 10, 10])  # 1~6이 나온 횟수
total = observed.sum()

# 기대 빈도: 공평한 주사위라면 각 면이 60/6 = 10번씩 나와야 함
expected = np.full(6, total / 6)

print(f"면:       1    2    3    4    5    6")
print(f"관측빈도: {observed}")
print(f"기대빈도: {expected}")
print()

# 카이제곱 적합도 검정
chi2, p_value = stats.chisquare(f_obs=observed)

print(f"카이제곱 통계량: {chi2:.4f}")
print(f"p-값:           {p_value:.4f}")
print()

if p_value <= 0.05:
    print(f"→ p={p_value:.4f} <= 0.05: H₀ 기각 → 주사위가 공평하지 않음")
else:
    print(f"→ p={p_value:.4f} ≥ 0.05: H₀ 기각 실패 → 주사위가 공평하다고 볼 수 있음")

### 4-2. 독립성 검정 (Independence Test)

시나리오: **"성별과 선호 음료(커피/차/주스)는 관계없는가?"**

- H₀: 성별과 선호 음료는 독립 (관계없음)
- H₁: 성별과 선호 음료는 관련이 있음

In [ ]:
# 분할표 (2×3): 행=성별(남/여), 열=음료(커피/차/주스)
table = np.array([[30, 10, 20],   # 남성: 커피 30, 차 10, 주스 20
                  [15, 25, 20]])  # 여성: 커피 15, 차 25, 주스 20

df_ct = pd.DataFrame(table,
                     index=['남성', '여성'],
                     columns=['커피', '차', '주스'])
print("분할표 (Contingency Table):")
print(df_ct)
print()

# 카이제곱 독립성 검정
chi2, p_value, dof, expected = stats.chi2_contingency(table)

print(f"카이제곱 통계량: {chi2:.4f}")
print(f"p-값:           {p_value:.4f}")
print(f"자유도:          {dof}")
print()
print("기대 빈도 (Expected):")
print(pd.DataFrame(expected, index=['남성', '여성'], columns=['커피', '차', '주스']).round(2))
print()

if p_value <= 0.05:
    print(f"→ p={p_value:.4f} <= 0.05: H₀ 기각 → 성별과 선호 음료는 관련이 있음")
else:
    print(f"→ p={p_value:.4f} > 0.05: H₀ 기각 실패 → 성별과 선호 음료는 독립")

---
## + PART 5. 비모수 검정

정규성 가정을 만족하지 않거나 표본 크기가 매우 작을 때 사용하는 검정.  
모수(파라미터)에 대한 가정 없이 순위(rank) 기반으로 비교.

### 5-1. Mann-Whitney U 검정

독립 표본 t-검정의 비모수 버전.  
시나리오: 정규성 가정이 어려운 두 집단의 분포 비교.

In [ ]:
# 3-3에서 사용한 A반, B반 데이터 재사용
u_stat, p_value = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')

print(f"Mann-Whitney U 검정:")
print(f"  U-통계량: {u_stat:.4f}")
print(f"  p-값:     {p_value:.4f}")

if p_value <= 0.05:
    print("  → H₀ 기각: 두 집단의 분포에 유의한 차이가 있음")
else:
    print("  → H₀ 기각 실패: 유의한 차이 없음")

### 5-2. Wilcoxon 부호순위 검정

대응 표본 t-검정의 비모수 버전.

In [ ]:
# 3-4에서 사용한 수강 전후 데이터 재사용
w_stat, p_value = stats.wilcoxon(before, after)

print(f"Wilcoxon 부호순위 검정:")
print(f"  W-통계량: {w_stat:.4f}")
print(f"  p-값:     {p_value:.4f}")

if p_value <= 0.05:
    print("  → H₀ 기각: 수강 전후 점수에 유의한 차이가 있음")
else:
    print("  → H₀ 기각 실패: 유의한 차이 없음")

### 5-3. Kruskal-Wallis 검정

일원분산분석(ANOVA)의 비모수 버전. 3개 이상 집단 비교.  
시나리오: **"A, B, C 세 반의 점수 분포에 차이가 있는가?"**

In [ ]:
# 세 반의 점수 데이터
class_a = np.array([72, 68, 75, 80, 65, 78, 70, 74, 69, 77])
class_b = np.array([65, 60, 68, 72, 58, 70, 63, 67, 62, 71])
class_c = np.array([80, 85, 78, 88, 82, 76, 84, 90, 79, 83])

print(f"A반 평균: {np.mean(class_a):.1f}")
print(f"B반 평균: {np.mean(class_b):.1f}")
print(f"C반 평균: {np.mean(class_c):.1f}")
print()

h_stat, p_value = stats.kruskal(class_a, class_b, class_c)

print(f"Kruskal-Wallis 검정:")
print(f"  H-통계량: {h_stat:.4f}")
print(f"  p-값:     {p_value:.4f}")

if p_value <= 0.05:
    print("  → H₀ 기각: 최소 한 반 이상 점수 분포에 유의한 차이가 있음")
else:
    print("  → H₀ 기각 실패: 세 반의 점수 분포에 유의한 차이 없음")

### 5-4. KS 검정 (정규성 검정)

데이터가 특정 분포(예: 정규분포)를 따르는지 검정.

- H₀: 데이터는 정규분포를 따른다
- H₁: 정규분포를 따르지 않는다

In [ ]:
# 시험 점수 데이터가 정규분포를 따르는지 확인
mean_s = np.mean(scores)
std_s = np.std(scores, ddof=1)

# KS 검정 (데이터를 표준화하거나 loc/scale 지정)
ks_stat, p_value = stats.kstest(scores, 'norm', args=(mean_s, std_s))

print(f"KS 정규성 검정:")
print(f"  KS-통계량: {ks_stat:.4f}")
print(f"  p-값:      {p_value:.4f}")

if p_value <= 0.05:
    print("  → H₀ 기각: 데이터가 정규분포를 따르지 않음")
else:
    print("  → H₀ 기각 실패: 정규분포를 따른다고 볼 수 있음")

---
## 요약

| 함수 | 검정 종류 | 데이터 유형 |
|------|----------|------------|
| `stats.ttest_1samp()` | 단일 표본 t-검정 | 연속형, 정규 가정 |
| `stats.ttest_ind()` | 독립 표본 t-검정 | 연속형, 정규 가정 |
| `stats.ttest_rel()` | 대응 표본 t-검정 | 연속형, 정규 가정 |
| `stats.chisquare()` | 카이제곱 적합도 검정 | 범주형 |
| `stats.chi2_contingency()` | 카이제곱 독립성 검정 | 범주형 |
| `stats.mannwhitneyu()` | Mann-Whitney U 검정 | 비모수, 독립 2집단 |
| `stats.wilcoxon()` | Wilcoxon 부호순위 검정 | 비모수, 대응 표본 |
| `stats.kruskal()` | Kruskal-Wallis 검정 | 비모수, 독립 3집단 이상 |
| `stats.kstest()` | KS 검정 (정규성) | 분포 적합 검정 |

**p-값 해석 원칙**
- **$p \leq \alpha$** → 귀무가설(H₀) **기각** → 통계적으로 유의한 차이/관계 있음
- **$p > \alpha$** → 귀무가설(H₀) **기각 실패** → 유의한 차이/관계를 발견하지 못함